# Exercise 1 - 3D - 2D

In [1]:
import numpy as np
import cv2 as cv2
from numpy.linalg import inv, pinv
import matplotlib.pyplot as plt

path = "C:/Users/peisz/OneDrive/Documents/Egyetem/DTU/3rd semester/PAS/Week 9/"

Recall from the slides the steps from Algorithm 3:

![title](algorithm_3.png)

![title](PnP.png)

# Exercise 1a)
The steps 1)-2.1) has already been done, and is saved in corresponding files. The exercise is to implement step 2.2) by filling in the missing code below

In [5]:
def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Use OpenCV to find the prev_points from the prev_img in the next_img
    Remember to remove points that could not be found from prev_points, next_points, and world_points
    hint: status == 1
    """
    # Parameters for Lucas-Kanade Optical Flow
    # winSize: Size of the search window at each pyramid level.
    # maxLevel: 0-based maximal pyramid level number (0 = original image only, 3 = 3 extra layers).
    # criteria: Termination criteria for the iterative search algorithm.
    params = dict(winSize=(21, 21),
                  maxLevel=3,
                  criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    
    # Calculate Optical Flow
    # next_points: The calculated new positions of the input features in the next image.
    # status: 1 if the flow for the corresponding feature has been found, 0 otherwise.
    # err: The error measure for the tracking.
    next_points, status, _ = cv2.calcOpticalFlowPyrLK(prev_img, next_img, prev_points, None, **params)

    if status is None:
        return None, None, None

    # Filtering step:
    # We must only keep points that were successfully tracked (status == 1).
    # .ravel() flattens the array to 1D.
    # status == 1 creates a boolean mask.
    good_status = status.ravel() == 1
    
    # Apply mask to all arrays to keep them synchronized
    next_points = next_points[good_status]
    prev_points = prev_points[good_status]
    world_points = world_points[good_status]

    return world_points, prev_points, next_points

Hint: Exercise 4 in week 2

# Exercise 1b)
Continue the algorithm by implementing step 2.3)

In [6]:
# Camera Matrix (Intrinsics): fx, fy, cx, cy
K = np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02], 
              [0, 7.188560e+02, 1.852157e+02],
              [0, 0, 1]])

reference_img = np.load(path + "img_" + str(0) + ".npy")

for t in range(1, 6):
    # the image at current time=t
    curImage = np.load(path + "img_" + str(t) + ".npy")
    # the 3D landmarks in the world coordinates which have been computed in time=t-1
    landmark_3D = np.load(path + "landmark_" + str(t-1) + ".npy")
    # the 2D coordinates of the 3D points in the previous frame at time=t-1
    reference_2D = np.load(path + "reference_2D_" + str(t-1) + ".npy")
    
    # Track features from the previous frame to the current frame.
    # This gives us the 2D correspondence: Where did the 3D landmark appear in the new image?
    landmark_3D, reference_2D, tracked_2Dpoints = featureTracking(reference_img, 
                                                                  curImage, 
                                                                  reference_2D,
                                                                  landmark_3D)
    
    """
    Using OpenCV, implement PnP using Ransac
    """
    # cv2.solvePnPRansac(objectPoints, imagePoints, cameraMatrix, distCoeffs)
    # This function estimates the object pose (rotation and translation) that minimizes the reprojection error.
    # objectPoints: Array of object points in the object coordinate space (3D).
    # imagePoints: Array of corresponding image points (2D).
    # K: Camera matrix.
    # Returns:
    # rvec: Rotation vector (Rodrigues format).
    # tvec: Translation vector.
    # inliers: Indices of points that fit the model well.
    _, rvec, tvec, inliers = cv2.solvePnPRansac(landmark_3D, tracked_2Dpoints, K, None)


    """
    Transform the translation and rotation into the world frame
    """
    # cv2.solvePnP returns the pose of the World relative to the Camera.
    # Transformation: P_cam = R * P_world + t
    # To get the Camera position in the World frame, we need to invert this:
    # P_world = R^T * (P_cam - t)
    # The camera center is at P_cam = (0,0,0), so Camera_World_Pos = -R^T * t
    
    # Convert Rotation Vector to Rotation Matrix
    R, _ = cv2.Rodrigues(rvec)
    
    # Compute camera position in world coordinates
    t = -R.T @ tvec
    
    
    print(tvec[0], tvec[1], tvec[2], rvec[0], rvec[1], rvec[2])

    # update for next timestep
    reference_img = curImage

[0.0011027] [0.0006718] [0.00078346] [-7.40069215e-05] [-7.35119066e-05] [9.84544281e-05]
[0.00145688] [0.00728043] [-0.67583403] [-0.00216658] [0.00325854] [-0.00244333]
[0.00062521] [0.01132867] [-1.37750276] [-0.00364614] [0.00751509] [-0.00099692]
[0.00803008] [0.01488408] [-2.10000505] [-0.00509583] [0.01121646] [-0.00082978]
[0.00399554] [0.01942941] [-2.83359408] [-0.00561424] [0.0161333] [0.00041981]


Hint: The output should look similar to:

[-0.00110282] [-0.00067164] [-0.00078343] [-7.40069212e-05] [-7.35119065e-05] [9.84544279e-05]

[-0.00363946] [-0.00875075] [0.67580842] [-0.0021666] [0.00325853] [-0.00244333]

[-0.01096271] [-0.01635663] [1.3774094] [-0.00364615] [0.0075151] [-0.00099691]

[-0.0315663] [-0.02560111] [2.0996797] [-0.00509583] [0.01121646] [-0.00082978]

[-0.04971858] [-0.03532535] [2.8330071] [-0.00561424] [0.0161333] [0.00041981]

# Exercise 1c)
What approximate direction did the camera move in?

In [7]:
print("Rotation:")
print(R)
print("Translation:")
print(t)
# Looking at the output of 't' (and 'tvec' above), notice which coordinate changes the most.
# The 3rd component (Z) increases significantly (0.6 -> 1.3 -> 2.0 -> 2.8).
# This implies the camera is moving primarily forward along the Z-axis.

Rotation:
[[ 9.99869773e-01 -4.65077607e-04  1.61313384e-02]
 [ 3.74503562e-04  9.99984152e-01  5.61735441e-03]
 [-1.61336953e-02 -5.61058164e-03  9.99854102e-01]]
Translation:
[[-0.04971864]
 [-0.03532535]
 [ 2.83300707]]
